In [1]:
import pandas as pd
import numpy as np
import pickle
import warnings
import os
warnings.filterwarnings('ignore')

import xgboost as xgb
from sklearn.ensemble import (RandomForestClassifier, ExtraTreesClassifier,
                              HistGradientBoostingClassifier)
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (classification_report, roc_auc_score, accuracy_score,
                             f1_score, average_precision_score)
from sklearn.model_selection import cross_val_score
from imblearn.pipeline import Pipeline as ImbPipeline   # ADDED
from imblearn.combine import SMOTETomek                  # ADDED
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

with open('C:/maternal_health_project/models/train_test_data.pkl','rb') as f:
    d = pickle.load(f)

X_train, y_train = d['X_train_bal'], d['y_train_bal']
X_train_pre, y_train_pre = d['X_train_pre_smote'], d['y_train_pre_smote']   # ADDED — pre-SMOTE, for tuning only
X_cal,   y_cal    = d['X_cal'],  d['y_cal']
X_test,  y_test   = d['X_test'], d['y_test']
feature_cols      = d['feature_cols']

y_train_arr = y_train.values if hasattr(y_train,'values') else y_train
y_cal_arr   = y_cal.values   if hasattr(y_cal,'values')   else y_cal
y_test_arr  = y_test.values  if hasattr(y_test,'values')  else y_test

print("✅ Data loaded — BINARY classification, leakage-free splits")
print(f"X_train (balanced): {X_train.shape} | X_train_pre (for CV): {X_train_pre.shape} | "
      f"X_cal: {X_cal.shape} | X_test: {X_test.shape}")
print(f"High Risk rate — train_pre: {y_train_pre.mean()*100:.2f}% (real) | "
      f"cal: {y_cal_arr.mean()*100:.2f}% (real) | test: {y_test_arr.mean()*100:.2f}% (real)")

def find_best_threshold(y_true, y_prob):
    thresholds = np.linspace(0.05, 0.95, 181)
    scores = [f1_score(y_true, (y_prob >= t).astype(int), average='macro') for t in thresholds]
    best_idx = int(np.argmax(scores))
    return thresholds[best_idx], scores[best_idx]

def evaluate_binary(name, y_true, y_pred, y_prob_pos):
    auc = roc_auc_score(y_true, y_prob_pos)
    acc = accuracy_score(y_true, y_pred)
    f1  = f1_score(y_true, y_pred, average='macro')
    ap  = average_precision_score(y_true, y_prob_pos)
    print(f"\n{name}")
    print(f"  AUC-ROC:  {auc:.4f}")
    print(f"  Accuracy: {acc:.4f}")
    print(f"  F1-Macro: {f1:.4f}")
    print(f"  Avg Prec: {ap:.4f}")
    print(classification_report(y_true, y_pred, target_names=['Not High Risk','High Risk']))
    return auc, acc, f1, ap

✅ Data loaded — BINARY classification, leakage-free splits
X_train (balanced): (187084, 21) | X_train_pre (for CV): (113179, 21) | X_cal: (28295, 21) | X_test: (35369, 21)
High Risk rate — train_pre: 16.50% (real) | cal: 16.50% (real) | test: 16.50% (real)


In [2]:
print("Tuning XGBoost (30 trials, SMOTE applied inside each CV fold)...")

def objective_xgb(trial):
    params = {
        'n_estimators':     trial.suggest_int('n_estimators', 200, 800),
        'max_depth':        trial.suggest_int('max_depth', 4, 10),
        'learning_rate':    trial.suggest_float('learning_rate', 0.01, 0.2),
        'subsample':        trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'eval_metric': 'logloss', 'random_state': 42, 'n_jobs': -1
    }
    pipe = ImbPipeline([
        ('smote', SMOTETomek(random_state=42)),
        ('clf', xgb.XGBClassifier(**params))
    ])
    return cross_val_score(pipe, X_train_pre, y_train_pre, cv=3, scoring='roc_auc').mean()

study_xgb = optuna.create_study(direction='maximize')
study_xgb.optimize(objective_xgb, n_trials=30, show_progress_bar=True)
print(f"Best XGBoost CV AUC (leak-free): {study_xgb.best_value:.4f}")

xgb_model = xgb.XGBClassifier(**study_xgb.best_params, eval_metric='logloss', random_state=42, n_jobs=-1)
xgb_model.fit(X_train, y_train)   # final fit still uses the SMOTE-balanced training set

xgb_cal_prob = xgb_model.predict_proba(X_cal)[:,1]
xgb_thresh, xgb_cal_f1 = find_best_threshold(y_cal_arr, xgb_cal_prob)
print(f"Tuned threshold: {xgb_thresh:.2f} (cal Macro-F1: {xgb_cal_f1:.4f})")

xgb_prob = xgb_model.predict_proba(X_test)[:,1]
xgb_pred = (xgb_prob >= xgb_thresh).astype(int)
xgb_auc, xgb_acc, xgb_f1, xgb_ap = evaluate_binary('XGBoost', y_test_arr, xgb_pred, xgb_prob)

with open('C:/maternal_health_project/models/xgb_model.pkl','wb') as f:
    pickle.dump({'model': xgb_model, 'threshold': xgb_thresh}, f)
print("✅ XGBoost saved.")

Tuning XGBoost (30 trials, SMOTE applied inside each CV fold)...


  0%|          | 0/30 [00:00<?, ?it/s]

Best XGBoost CV AUC (leak-free): 0.7410
Tuned threshold: 0.33 (cal Macro-F1: 0.6912)

XGBoost
  AUC-ROC:  0.7533
  Accuracy: 0.8425
  F1-Macro: 0.6911
  Avg Prec: 0.4468
               precision    recall  f1-score   support

Not High Risk       0.89      0.92      0.91     29532
    High Risk       0.53      0.43      0.47      5837

     accuracy                           0.84     35369
    macro avg       0.71      0.68      0.69     35369
 weighted avg       0.83      0.84      0.84     35369

✅ XGBoost saved.


In [3]:
print("Tuning Random Forest (30 trials, SMOTE applied inside each CV fold)...")

def objective_rf(trial):
    params = {
        'n_estimators':      trial.suggest_int('n_estimators', 200, 600),
        'max_depth':         trial.suggest_int('max_depth', 5, 25),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 20),
        'min_samples_leaf':  trial.suggest_int('min_samples_leaf', 1, 10),
        'max_features':      trial.suggest_categorical('max_features', ['sqrt','log2']),
        'random_state': 42, 'n_jobs': -1
    }
    pipe = ImbPipeline([
        ('smote', SMOTETomek(random_state=42)),
        ('clf', RandomForestClassifier(**params))
    ])
    return cross_val_score(pipe, X_train_pre, y_train_pre, cv=3, scoring='roc_auc').mean()

study_rf = optuna.create_study(direction='maximize')
study_rf.optimize(objective_rf, n_trials=30, show_progress_bar=True)
print(f"Best RF CV AUC (leak-free): {study_rf.best_value:.4f}")

rf_model = RandomForestClassifier(**study_rf.best_params, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)

rf_cal_prob = rf_model.predict_proba(X_cal)[:,1]
rf_thresh, rf_cal_f1 = find_best_threshold(y_cal_arr, rf_cal_prob)
print(f"Tuned threshold: {rf_thresh:.2f} (cal Macro-F1: {rf_cal_f1:.4f})")

rf_prob = rf_model.predict_proba(X_test)[:,1]
rf_pred = (rf_prob >= rf_thresh).astype(int)
rf_auc, rf_acc, rf_f1, rf_ap = evaluate_binary('Random Forest', y_test_arr, rf_pred, rf_prob)

with open('C:/maternal_health_project/models/rf_model.pkl','wb') as f:
    pickle.dump({'model': rf_model, 'threshold': rf_thresh}, f)
print("✅ Random Forest saved.")

Tuning Random Forest (30 trials, SMOTE applied inside each CV fold)...


  0%|          | 0/30 [00:00<?, ?it/s]

Best RF CV AUC (leak-free): 0.7248
Tuned threshold: 0.46 (cal Macro-F1: 0.6752)

Random Forest
  AUC-ROC:  0.7349
  Accuracy: 0.8324
  F1-Macro: 0.6720
  Avg Prec: 0.4142
               precision    recall  f1-score   support

Not High Risk       0.89      0.92      0.90     29532
    High Risk       0.49      0.40      0.44      5837

     accuracy                           0.83     35369
    macro avg       0.69      0.66      0.67     35369
 weighted avg       0.82      0.83      0.83     35369

✅ Random Forest saved.


In [4]:
print("Tuning SVM (15 trials — SVM scales slower with data size, keep trials modest)...")

from sklearn.svm import SVC

def objective_svm(trial):
    params = {
        'C':       trial.suggest_float('C', 0.1, 10.0, log=True),
        'gamma':   trial.suggest_float('gamma', 1e-4, 1e-1, log=True),
        'kernel':  'rbf',
        'probability': True,
        'random_state': 42
    }
    pipe = ImbPipeline([
        ('smote', SMOTETomek(random_state=42)),
        ('scaler', StandardScaler()),
        ('clf', SVC(**params))
    ])
    # Subsample just for tuning speed — SVM training time grows fast with n
    sample_idx = np.random.RandomState(42).choice(
        len(X_train_pre), size=min(8000, len(X_train_pre)), replace=False)
    return cross_val_score(
        pipe, X_train_pre.iloc[sample_idx], y_train_pre.iloc[sample_idx],
        cv=3, scoring='roc_auc'
    ).mean()

study_svm = optuna.create_study(direction='maximize')
study_svm.optimize(objective_svm, n_trials=15, show_progress_bar=True)
print(f"Best SVM CV AUC (leak-free, subsampled): {study_svm.best_value:.4f}")

# Subsample the FINAL training fit too — kernel SVM does not scale to 100K+ rows in reasonable time.
# 15,000 rows is still a substantial, representative training set.
max_svm_train = 15000
if len(X_train) > max_svm_train:
    sub_idx = np.random.RandomState(42).choice(len(X_train), size=max_svm_train, replace=False)
    X_train_svm_fit = X_train.iloc[sub_idx]
    y_train_svm_fit = y_train.iloc[sub_idx] if hasattr(y_train, 'iloc') else y_train[sub_idx]
    print(f"Subsampled final SVM training set: {len(X_train):,} -> {max_svm_train:,} rows")
else:
    X_train_svm_fit = X_train
    y_train_svm_fit = y_train

scaler_svm = StandardScaler()
X_train_scaled = scaler_svm.fit_transform(X_train_svm_fit)
X_cal_scaled   = scaler_svm.transform(X_cal)
X_test_scaled  = scaler_svm.transform(X_test)

svm_model = SVC(**study_svm.best_params, probability=True, random_state=42)
svm_model.fit(X_train_scaled, y_train_svm_fit)
print("✅ SVM final fit complete.")

svm_cal_prob = svm_model.predict_proba(X_cal_scaled)[:,1]
svm_thresh, svm_cal_f1 = find_best_threshold(y_cal_arr, svm_cal_prob)
print(f"Tuned threshold: {svm_thresh:.2f} (cal Macro-F1: {svm_cal_f1:.4f})")

svm_prob = svm_model.predict_proba(X_test_scaled)[:,1]
svm_pred = (svm_prob >= svm_thresh).astype(int)
svm_auc, svm_acc, svm_f1, svm_ap = evaluate_binary('SVM', y_test_arr, svm_pred, svm_prob)

with open('C:/maternal_health_project/models/svm_model.pkl','wb') as f:
    pickle.dump({'model': svm_model, 'threshold': svm_thresh, 'scaler': scaler_svm}, f)
print("✅ SVM saved.")

Tuning SVM (15 trials — SVM scales slower with data size, keep trials modest)...


  0%|          | 0/15 [00:00<?, ?it/s]

Best SVM CV AUC (leak-free, subsampled): 0.7004
Subsampled final SVM training set: 187,084 -> 15,000 rows
✅ SVM final fit complete.
Tuned threshold: 0.71 (cal Macro-F1: 0.6488)

SVM
  AUC-ROC:  0.7082
  Accuracy: 0.8129
  F1-Macro: 0.6476
  Avg Prec: 0.3670
               precision    recall  f1-score   support

Not High Risk       0.88      0.90      0.89     29532
    High Risk       0.43      0.39      0.41      5837

     accuracy                           0.81     35369
    macro avg       0.65      0.64      0.65     35369
 weighted avg       0.81      0.81      0.81     35369

✅ SVM saved.


In [5]:
print("Tuning HistGradientBoosting (30 trials, SMOTE applied inside each CV fold)...")

def objective_hgb(trial):
    params = {
        'max_iter':          trial.suggest_int('max_iter', 100, 500),
        'max_depth':         trial.suggest_int('max_depth', 3, 10),
        'learning_rate':     trial.suggest_float('learning_rate', 0.01, 0.2),
        'min_samples_leaf':  trial.suggest_int('min_samples_leaf', 10, 100),
        'l2_regularization': trial.suggest_float('l2_regularization', 0, 1.0),
        'max_leaf_nodes':    trial.suggest_int('max_leaf_nodes', 15, 63),
        'random_state': 42
    }
    pipe = ImbPipeline([
        ('smote', SMOTETomek(random_state=42)),
        ('clf', HistGradientBoostingClassifier(**params, class_weight='balanced'))
    ])
    return cross_val_score(pipe, X_train_pre, y_train_pre, cv=3, scoring='roc_auc').mean()

study_hgb = optuna.create_study(direction='maximize')
study_hgb.optimize(objective_hgb, n_trials=30, show_progress_bar=True)
print(f"Best HGB CV AUC (leak-free): {study_hgb.best_value:.4f}")

hgb_model = HistGradientBoostingClassifier(**study_hgb.best_params, class_weight='balanced', random_state=42)
hgb_model.fit(X_train, y_train)

hgb_cal_prob = hgb_model.predict_proba(X_cal)[:,1]
hgb_thresh, hgb_cal_f1 = find_best_threshold(y_cal_arr, hgb_cal_prob)
print(f"Tuned threshold: {hgb_thresh:.2f} (cal Macro-F1: {hgb_cal_f1:.4f})")

hgb_prob = hgb_model.predict_proba(X_test)[:,1]
hgb_pred = (hgb_prob >= hgb_thresh).astype(int)
hgb_auc, hgb_acc, hgb_f1, hgb_ap = evaluate_binary('HistGradientBoosting', y_test_arr, hgb_pred, hgb_prob)

with open('C:/maternal_health_project/models/hgb_model.pkl','wb') as f:
    pickle.dump({'model': hgb_model, 'threshold': hgb_thresh}, f)
print("✅ HistGradientBoosting saved.")

Tuning HistGradientBoosting (30 trials, SMOTE applied inside each CV fold)...


  0%|          | 0/30 [00:00<?, ?it/s]

Best HGB CV AUC (leak-free): 0.7390
Tuned threshold: 0.34 (cal Macro-F1: 0.6935)

HistGradientBoosting
  AUC-ROC:  0.7511
  Accuracy: 0.8443
  F1-Macro: 0.6909
  Avg Prec: 0.4453
               precision    recall  f1-score   support

Not High Risk       0.89      0.93      0.91     29532
    High Risk       0.54      0.42      0.47      5837

     accuracy                           0.84     35369
    macro avg       0.71      0.68      0.69     35369
 weighted avg       0.83      0.84      0.84     35369

✅ HistGradientBoosting saved.


In [6]:
print("Tuning Extra Trees (30 trials, SMOTE applied inside each CV fold)...")

def objective_et(trial):
    params = {
        'n_estimators':      trial.suggest_int('n_estimators', 200, 600),
        'max_depth':         trial.suggest_int('max_depth', 5, 25),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 20),
        'min_samples_leaf':  trial.suggest_int('min_samples_leaf', 1, 10),
        'max_features':      trial.suggest_categorical('max_features', ['sqrt','log2']),
        'random_state': 42, 'n_jobs': -1
    }
    pipe = ImbPipeline([
        ('smote', SMOTETomek(random_state=42)),
        ('clf', ExtraTreesClassifier(**params))
    ])
    return cross_val_score(pipe, X_train_pre, y_train_pre, cv=3, scoring='roc_auc').mean()

study_et = optuna.create_study(direction='maximize')
study_et.optimize(objective_et, n_trials=30, show_progress_bar=True)
print(f"Best ET CV AUC (leak-free): {study_et.best_value:.4f}")

et_model = ExtraTreesClassifier(**study_et.best_params, random_state=42, n_jobs=-1)
et_model.fit(X_train, y_train)

et_cal_prob = et_model.predict_proba(X_cal)[:,1]
et_thresh, et_cal_f1 = find_best_threshold(y_cal_arr, et_cal_prob)
print(f"Tuned threshold: {et_thresh:.2f} (cal Macro-F1: {et_cal_f1:.4f})")

et_prob = et_model.predict_proba(X_test)[:,1]
et_pred = (et_prob >= et_thresh).astype(int)
et_auc, et_acc, et_f1, et_ap = evaluate_binary('Extra Trees', y_test_arr, et_pred, et_prob)

with open('C:/maternal_health_project/models/et_model.pkl','wb') as f:
    pickle.dump({'model': et_model, 'threshold': et_thresh}, f)
print("✅ Extra Trees saved.")

Tuning Extra Trees (30 trials, SMOTE applied inside each CV fold)...


  0%|          | 0/30 [00:00<?, ?it/s]

Best ET CV AUC (leak-free): 0.7072
Tuned threshold: 0.57 (cal Macro-F1: 0.6535)

Extra Trees
  AUC-ROC:  0.7172
  Accuracy: 0.8171
  F1-Macro: 0.6497
  Avg Prec: 0.3806
               precision    recall  f1-score   support

Not High Risk       0.88      0.90      0.89     29532
    High Risk       0.44      0.38      0.41      5837

     accuracy                           0.82     35369
    macro avg       0.66      0.64      0.65     35369
 weighted avg       0.81      0.82      0.81     35369

✅ Extra Trees saved.


In [7]:
print("Training Gaussian Naive Bayes (fundamentally different algorithm family)...")

from sklearn.naive_bayes import GaussianNB

nb_model = GaussianNB()
nb_model.fit(X_train, y_train)

nb_cal_prob = nb_model.predict_proba(X_cal)[:,1]
nb_thresh, nb_cal_f1 = find_best_threshold(y_cal_arr, nb_cal_prob)
print(f"Tuned threshold: {nb_thresh:.2f} (cal Macro-F1: {nb_cal_f1:.4f})")

nb_prob = nb_model.predict_proba(X_test)[:,1]
nb_pred = (nb_prob >= nb_thresh).astype(int)
nb_auc, nb_acc, nb_f1, nb_ap = evaluate_binary('Gaussian Naive Bayes', y_test_arr, nb_pred, nb_prob)

with open('C:/maternal_health_project/models/nb_model.pkl','wb') as f:
    pickle.dump({'model': nb_model, 'threshold': nb_thresh}, f)
print("✅ Naive Bayes saved.")

Training Gaussian Naive Bayes (fundamentally different algorithm family)...
Tuned threshold: 0.95 (cal Macro-F1: 0.6056)

Gaussian Naive Bayes
  AUC-ROC:  0.6842
  Accuracy: 0.7332
  F1-Macro: 0.6059
  Avg Prec: 0.3420
               precision    recall  f1-score   support

Not High Risk       0.89      0.78      0.83     29532
    High Risk       0.31      0.50      0.38      5837

     accuracy                           0.73     35369
    macro avg       0.60      0.64      0.61     35369
 weighted avg       0.79      0.73      0.76     35369

✅ Naive Bayes saved.


In [8]:
print("\n" + "="*70)
print("   INDIVIDUAL MODEL COMPARISON — BINARY CLASSIFICATION (tuned thresholds)")
print("="*70)
print(f"{'Model':<28} {'Accuracy':<12} {'F1-Macro':<12} {'AUC-ROC':<10} {'Threshold':<10}")
print("-"*70)
models_results = [
    ('XGBoost',              xgb_acc, xgb_f1, xgb_auc, xgb_thresh),
    ('Random Forest',        rf_acc,  rf_f1,  rf_auc,  rf_thresh),
    ('Extra Trees',          et_acc,  et_f1,  et_auc,  et_thresh),
    ('HistGradientBoosting', hgb_acc, hgb_f1, hgb_auc, hgb_thresh),
    ('SVM',                  svm_acc, svm_f1, svm_auc, svm_thresh),
    ('Naive Bayes',          nb_acc, nb_f1, nb_auc, nb_thresh),
]
for name, acc, f1, auc, th in models_results:
    print(f"{name:<28} {acc:<12.4f} {f1:<12.4f} {auc:<10.4f} {th:<10.2f}")
print("="*70)

comparison_df = pd.DataFrame({
    'Model':     [m[0] for m in models_results],
    'Accuracy':  [m[1] for m in models_results],
    'F1-Macro':  [m[2] for m in models_results],
    'AUC-ROC':   [m[3] for m in models_results],
    'Threshold': [m[4] for m in models_results],
}).round(4)

os.makedirs('C:/maternal_health_project/outputs', exist_ok=True)
comparison_df.to_csv('C:/maternal_health_project/outputs/individual_models.csv', index=False)
print("\n✅ Comparison table saved.")


   INDIVIDUAL MODEL COMPARISON — BINARY CLASSIFICATION (tuned thresholds)
Model                        Accuracy     F1-Macro     AUC-ROC    Threshold 
----------------------------------------------------------------------
XGBoost                      0.8425       0.6911       0.7533     0.33      
Random Forest                0.8324       0.6720       0.7349     0.46      
Extra Trees                  0.8171       0.6497       0.7172     0.57      
HistGradientBoosting         0.8443       0.6909       0.7511     0.34      
SVM                          0.8129       0.6476       0.7082     0.71      
Naive Bayes                  0.7332       0.6059       0.6842     0.95      

✅ Comparison table saved.


In [9]:
files = ['xgb_model.pkl','rf_model.pkl','et_model.pkl','hgb_model.pkl','svm_model.pkl']
print("Model file verification:")
print("-"*50)
for f in files:
    path = f'C:/maternal_health_project/models/{f}'
    if os.path.exists(path):
        size = os.path.getsize(path)
        print(f"✅ {f:<30} {size:>10,} bytes")
    else:
        print(f"❌ {f} NOT FOUND")
print("\n✅ Phase 3 complete. Ready for Phase 4.")

Model file verification:
--------------------------------------------------
✅ xgb_model.pkl                   1,370,972 bytes
✅ rf_model.pkl                   592,977,211 bytes
✅ et_model.pkl                   954,476,750 bytes
✅ hgb_model.pkl                     348,772 bytes
✅ svm_model.pkl                   2,086,535 bytes

✅ Phase 3 complete. Ready for Phase 4.
